In [1]:
import pandas as pd
import numpy as np
import xarray as xr

In [2]:
# Open Easter East Africa MAM monthly ML data
ml_data = pd.read_csv('data/ml_data_automation_test_v2/eastern_east_africa_MAM_ml_data_monthly.csv')

# Open chirps
chirps = xr.open_dataset('data/CHIRPS/chirps-v2.0.monthly.nc')

In [47]:
# Subset CHIRPS to the Eastern East Africa Region
chirps_EEA_OND = chirps.sel(latitude=slice(3.5, 8.5), longitude=slice(38, 50)).to_dataframe().reset_index()

# Add year column from time
chirps_EEA_OND['year'] = chirps_EEA_OND['time'].dt.year

# Add month column from time
chirps_EEA_OND['month'] = chirps_EEA_OND['time'].dt.month

# Drop time column from dataframe
chirps_EEA_OND = chirps_EEA_OND.drop(columns='time')

# Keep only values from months 10 to 12
chirps_EEA_OND = chirps_EEA_OND.query('10 <= month <= 12')

# Calculate Spatial Mean
chirps_EEA_OND = chirps_EEA_OND.groupby(['year', 'month'])['precip'].mean().reset_index()

# Align values from dataframe to corresponding temporal value of ML data
chirps_EEA_OND['year'] = chirps_EEA_OND['year'] + 1
chirps_EEA_OND['month'] = (chirps_EEA_OND['month'] + 5) % 12

# Rename precip column
chirps_EEA_OND = chirps_EEA_OND.rename({'precip': 'EEA_precip'}, axis=1)

# Normalize the precip column
mean_val = chirps_EEA_OND['EEA_precip'].mean()
std_val = chirps_EEA_OND['EEA_precip'].std()
chirps_EEA_OND['EEA_precip'] = (chirps_EEA_OND['EEA_precip'] - mean_val) / std_val

In [48]:
# Subset CHIRPS to the Eastern East Africa Region
chirps_SA_OND = chirps.sel(latitude=slice(-23, -15), longitude=slice(25, 34)).to_dataframe().reset_index()

# Add year column from time
chirps_SA_OND['year'] = chirps_SA_OND['time'].dt.year

# Add month column from time
chirps_SA_OND['month'] = chirps_SA_OND['time'].dt.month

# Drop time column from dataframe
chirps_SA_OND = chirps_SA_OND.drop(columns='time')

# Keep only values from months 10 to 12
chirps_SA_OND = chirps_SA_OND.query('10 <= month <= 12')

# Calculate Spatial Mean
chirps_SA_OND = chirps_SA_OND.groupby(['year', 'month'])['precip'].mean().reset_index()

# Align values from dataframe to corresponding temporal value of ML data
chirps_SA_OND['year'] = chirps_SA_OND['year'] + 1
chirps_SA_OND['month'] = (chirps_SA_OND['month'] + 5) % 12

# Rename precip column
chirps_SA_OND = chirps_SA_OND.rename({'precip': 'SA_precip'}, axis=1)

# Normalize the precip column
mean_val = chirps_SA_OND['SA_precip'].mean()
std_val = chirps_SA_OND['SA_precip'].std()
chirps_SA_OND['SA_precip'] = (chirps_SA_OND['SA_precip'] - mean_val) / std_val

In [50]:
# Merge both SA_OND and EEA_OND chirps dataframes to ml_data
ml_with_precip = ml_data.merge(chirps_SA_OND, on=['year', 'month'], how='right').merge(chirps_EEA_OND, on=['year', 'month'], how='right').dropna()

# Output dataframe as csv
ml_with_precip.to_csv('data/ml_EEA_MAM_precip_predictors/eastern_east_africa_MAM_ml_data_monthly.csv')